# Notebook 03 — Prosodic Features

Extract paralinguistic signals from each audio clip in `data/` using Parselmouth
(Praat in Python): duration, F0 statistics, intensity, voiced fraction, a rough
pause ratio, and a proxy speaking rate.

**Inputs**: `data/*.wav`, `data/*.mp3`
**Outputs**:
- `output/03_prosody_features.csv`
- `output/pitch_contour.png` (first sample, for sanity check)

CPU-only — no GPU required. Run after notebook 01 (which produces the file
list); independent of notebook 02.


## 1. Imports and paths


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import parselmouth

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

PITCH_FLOOR_HZ = 75.0
PITCH_CEILING_HZ = 600.0
TIME_STEP_S = 0.01


## 2. `extract_prosody` — per-utterance feature extraction

Definitions (also documented in `docs/methodology.md`):

- **F0 mean / std / range** over voiced frames only, with pitch floor 75 Hz and
  ceiling 600 Hz (covers male–female adult range).
- **Intensity mean / std** in dB across all frames.
- **Voiced fraction** = voiced frames ÷ total pitch frames.
- **Pause ratio** = fraction of frames with intensity below
  `mean − 1.5 × std` (a rough silence detector — adequate for relative
  comparison, not absolute pause measurement).
- **Speaking rate** = voiced frames per second (proxy; not phoneme rate).


In [ ]:
def extract_prosody(audio_path: Path) -> dict[str, Any]:
    snd = parselmouth.Sound(str(audio_path))
    duration = snd.get_total_duration()

    pitch = snd.to_pitch(
        time_step=TIME_STEP_S,
        pitch_floor=PITCH_FLOOR_HZ,
        pitch_ceiling=PITCH_CEILING_HZ,
    )
    f0_values = pitch.selected_array["frequency"]
    f0_voiced = f0_values[f0_values > 0]

    if f0_voiced.size > 0:
        f0_mean = float(np.mean(f0_voiced))
        f0_std = float(np.std(f0_voiced))
        f0_range = float(np.ptp(f0_voiced))
    else:
        f0_mean = f0_std = f0_range = 0.0

    intensity = snd.to_intensity(time_step=TIME_STEP_S)
    intensity_values = intensity.values[0]
    intensity_mean = float(np.mean(intensity_values))
    intensity_std = float(np.std(intensity_values))

    voiced_fraction = (
        float(f0_voiced.size / f0_values.size) if f0_values.size > 0 else 0.0
    )

    silence_thresh = intensity_mean - 1.5 * intensity_std
    pause_ratio = float(np.mean(intensity_values < silence_thresh))

    speaking_rate = (
        float(f0_voiced.size * TIME_STEP_S / duration) if duration > 0 else 0.0
    )

    return {
        "duration_sec": float(duration),
        "f0_mean_hz": f0_mean,
        "f0_std_hz": f0_std,
        "f0_range_hz": f0_range,
        "intensity_mean_db": intensity_mean,
        "intensity_std_db": intensity_std,
        "voiced_fraction": voiced_fraction,
        "pause_ratio": pause_ratio,
        "speaking_rate": speaking_rate,
    }


## 3. Run extraction over all audio files and save CSV

Auto-discover audio in `data/`. The `file` column stores **basename only**
(e.g., `sample_01.wav`) so it matches the `file` column in
`01_whisper_results.csv` and `02_llm_uncertainty.csv` for the merge in
notebook 04.


In [ ]:
audio_files: list[Path] = sorted(
    [p for ext in ("*.wav", "*.mp3") for p in DATA_DIR.glob(ext)]
)

if not audio_files:
    raise FileNotFoundError(
        f"No .wav or .mp3 files found in {DATA_DIR}. "
        "Add 5–20 short English clips and re-run."
    )

records: list[dict[str, Any]] = []
for path in audio_files:
    feats = extract_prosody(path)
    feats["file"] = path.name
    records.append(feats)

df = pd.DataFrame(records)
column_order = ["file"] + [c for c in df.columns if c != "file"]
df = df[column_order]

csv_path = OUTPUT_DIR / "03_prosody_features.csv"
df.to_csv(csv_path, index=False)

print(f"Saved {len(df)} rows to {csv_path.relative_to(REPO_ROOT)}")
df.head(10)


## 4. Pitch contour for the first sample

Visual sanity check: plot the F0 trajectory of the first audio file using the
same pitch settings as the extraction step (75–600 Hz). Unvoiced frames are
masked with `NaN` so they leave gaps in the contour.

Output: `output/pitch_contour.png`.


In [ ]:
first_audio = audio_files[0]
snd = parselmouth.Sound(str(first_audio))
pitch = snd.to_pitch(
    time_step=TIME_STEP_S,
    pitch_floor=PITCH_FLOOR_HZ,
    pitch_ceiling=PITCH_CEILING_HZ,
)
times = pitch.xs()
freqs = pitch.selected_array["frequency"]
freqs_plot = np.where(freqs > 0, freqs, np.nan)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(times, freqs_plot, ".", markersize=2)
ax.set_xlabel("Time (s)")
ax.set_ylabel("F0 (Hz)")
ax.set_ylim(PITCH_FLOOR_HZ, PITCH_CEILING_HZ)
ax.set_title(f"Pitch contour — {first_audio.name}")
fig.tight_layout()

png_path = OUTPUT_DIR / "pitch_contour.png"
fig.savefig(png_path, dpi=120)
plt.show()

print(f"Saved pitch contour to {png_path.relative_to(REPO_ROOT)}")
